In [14]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import torchaudio 
import torchaudio.transforms as T

from datasets import load_dataset

import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from corpus.single_word import SingleWordDataset as Dataset

import fuzzy 
from tqdm.notebook import tqdm
import pandas as pd

In [7]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words']  # Name of data splits to be used as validation set


In [8]:
dataset = Dataset(path, dev_split, None, 1)


Using custom data configuration default-6e977ecf05190e74
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)


In [9]:
sampling_rate = 16000

In [10]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Check performance on demo dataset

In [11]:
dataset2 = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation")
dataset2 = dataset2.sort("id")
sampling_rate = dataset2.features["audio"].sampling_rate

Reusing dataset librispeech_asr (/home/imgriff/.cache/huggingface/datasets/hf-internal-testing___librispeech_asr/clean/2.1.0/d3bc4c2bc2078fcde3ad0f0f635862e4c0fef78ba94c4a34c4c250a097af240b)
Loading cached sorted indices for dataset at /home/imgriff/.cache/huggingface/datasets/hf-internal-testing___librispeech_asr/clean/2.1.0/d3bc4c2bc2078fcde3ad0f0f635862e4c0fef78ba94c4a34c4c250a097af240b/cache-2f7c0cbee6ef3aa1.arrow


In [21]:
sampling_rate

16000

In [23]:
# results = []

model = model.cuda()
for ix in tqdm(range(5)):
    true = dataset2[ix]["text"]
    inputs = processor(dataset2[ix]["audio"]['array'], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
    print(f'guess: {transcription[0]}')
    print(f'truth: {true}')
    print()
    
#     results.append({'hyp': transcription[0],
#                     'truth': true})

  0%|          | 0/5 [00:00<?, ?it/s]

guess: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
truth: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL

guess: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
truth: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER

guess: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND
truth: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND

guess: HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LAYTON'S WORK IS REALLY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA
truth: HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LEIGHTON'S WORK IS REALLY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA

guess: LINNELL'S PIC

We can see this is a character model based on the handful of errors made. Overall performance looks good.

## Check on Single Word Corpora

no language model is getting used, so things may be messy 

In [24]:
dataset.dataset

Dataset({
    features: ['Unnamed: 0', 'word', 'wav_path'],
    num_rows: 200
})

In [25]:
# resampler = T.Resample(44100, 16000, lowpass_filter_width=64,
#                    rolloff=0.9475937167399596, 
#                    resampling_method="kaiser_window",beta=14.769656459379492, dtype=torch.float32)

def load_audio(example):
    wav, sr = torchaudio.load(example['wav_path'])
#     wav = resampler(wav)
    example['audio'] = wav
    example['sampling_rate'] = 16000
    return example


In [26]:
updated_dataset = dataset.dataset.map(load_audio)

Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c28c661dad041df8.arrow


In [27]:
results = []

# model = model
for sample in tqdm(updated_dataset):
    true = sample["word"]
    inputs = processor(sample["audio"], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
#     print(f'guess: {transcription[0]}')
#     print(f'truth: {true}')
#     print()
    
    results.append({'hyp': transcription[0],
                    'truth': true})

  0%|          | 0/200 [00:00<?, ?it/s]

In [28]:
import pandas as pd

In [29]:
results = pd.DataFrame(results)

In [30]:
results.head(5)

,hyp,truth
0,THE,THE
1,AND,AND
2,TWO,TO
3,OF,OF
4,A,A


In [31]:
top_match_guesses = results[(results['hyp'] == results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses

159 matched guesses; 79.5% correct


,hyp,truth
0,THE,THE
1,AND,AND
3,OF,OF
4,A,A
5,THAT,THAT
...,...,...
191,BETTER,BETTER
194,BIG,BIG
195,TWENTY,TWENTY
196,EACH,EACH


In [33]:
# For homophone matching 

dmeta = fuzzy.DMetaphone()

results['hyp_dmeta'] = results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results['truth_dmeta'] = results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses

179 matched guesses; 89.5% correct


,hyp,truth,hyp_dmeta,truth_dmeta
0,THE,THE,"[b'0', b'T']","[b'0', b'T']"
1,AND,AND,"[b'ANT', None]","[b'ANT', None]"
2,TWO,TO,"[b'T', None]","[b'T', None]"
3,OF,OF,"[b'AF', None]","[b'AF', None]"
4,A,A,"[b'A', None]","[b'A', None]"
...,...,...,...,...
194,BIG,BIG,"[b'PK', None]","[b'PK', None]"
195,TWENTY,TWENTY,"[b'TNT', None]","[b'TNT', None]"
196,EACH,EACH,"[b'AK', None]","[b'AK', None]"
197,SURE,SURE,"[b'SR', None]","[b'SR', None]"


## On noise - 0 dBSNR

In [34]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_0SNR']  # Name of data splits to be used as validation set

dataset = Dataset(path, dev_split, None, 1)

updated_dataset = dataset.dataset.map(load_audio)

Using custom data configuration default-c9dd0aeaa110b3ae
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-1aa1c950d7d80a14.arrow


In [35]:
results = []

for sample in tqdm(updated_dataset):
    true = sample["word"]
    inputs = processor(sample["audio"], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
#     print(f'guess: {transcription[0]}')
#     print(f'truth: {true}')
#     print()
    
    results.append({'hyp': transcription[0],
                    'truth': true})

  0%|          | 0/200 [00:00<?, ?it/s]

In [36]:
results = pd.DataFrame(results)

top_match_guesses = results[(results['hyp'] == results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

119 matched guesses; 59.5% correct


,hyp,truth
1,AND,AND
5,THAT,THAT
7,I,I
9,IS,IS
10,IT,IT


In [37]:

results['hyp_dmeta'] = results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results['truth_dmeta'] = results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

141 matched guesses; 70.5% correct


,hyp,truth,hyp_dmeta,truth_dmeta
0,THO,THE,"[b'0', b'T']","[b'0', b'T']"
1,AND,AND,"[b'ANT', None]","[b'ANT', None]"
2,TWO,TO,"[b'T', None]","[b'T', None]"
5,THAT,THAT,"[b'0T', b'TT']","[b'0T', b'TT']"
7,I,I,"[b'A', None]","[b'A', None]"


## On noise at -5 dBSNR

In [40]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_-5SNR']  # Name of data splits to be used as validation set

dataset = Dataset(path, dev_split, None, 1)

updated_dataset = dataset.dataset.map(load_audio)

Using custom data configuration default-5683877c4c4495f8


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Dataset csv downloaded and prepared to /home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e. Subsequent calls will reuse this data.


0ex [00:00, ?ex/s]

In [41]:
results = []

for sample in tqdm(updated_dataset):
    true = sample["word"]
    inputs = processor(sample["audio"], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
#     print(f'guess: {transcription[0]}')
#     print(f'truth: {true}')
#     print()
    
    results.append({'hyp': transcription[0],
                    'truth': true})

  0%|          | 0/200 [00:00<?, ?it/s]

In [42]:
results = pd.DataFrame(results)

top_match_guesses = results[(results['hyp'] == results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

54 matched guesses; 27.0% correct


,hyp,truth
2,TO,TO
11,WAS,WAS
13,SO,SO
15,WE,WE
21,AS,AS


In [43]:

results['hyp_dmeta'] = results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
results['truth_dmeta'] = results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = results[(results['hyp_dmeta'] == results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

74 matched guesses; 37.0% correct


,hyp,truth,hyp_dmeta,truth_dmeta
2,TO,TO,"[b'T', None]","[b'T', None]"
9,AS,IS,"[b'AS', None]","[b'AS', None]"
11,WAS,WAS,"[b'AS', b'FS']","[b'AS', b'FS']"
13,SO,SO,"[b'S', None]","[b'S', None]"
15,WE,WE,"[b'A', b'F']","[b'A', b'F']"
